# 第2回 演習: 用語集を LoRA で学習し、学習前後を比較する

用語集の SFT データと、Notebook 内に置いた用語集にない語への回答例でモデルを LoRA 学習し、学習前後を比べます。評価では見出し語・言い換え・未登録語を尋ね、JSON 形式、`needs_review`、回答内容を確認します。

**この演習で確かめること**

1. 用語集にある項目では `needs_review: false` と定義を回答できるか。
2. 質問の言い回しを変えても、用語集の定義に沿って JSON で回答できるか。
3. 用語集にない項目では、推測せず `needs_review: true` と回答できるか。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

`needs_review` は「モデル自身がその概念を知らない」という意味ではなく、「この用語集に記載がない」という意味で使います。glossary_sft.jsonl の項目はすべて needs_review=false として扱い、未登録語の SFT 例はこの Notebook 内に置きます。評価用の未知語とは別の語を学習に使い、未見の未知語への一般化を測ります。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、用語集と評価データ、学習前評価 | 約 5 分 |
| 4〜7 | SFT データの読み込みと学習、学習後評価、結果の提出 | 学習データ量に依存 |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）、`Qwen/Qwen3.5-0.8B`（Apache-2.0）、`glossary_sft.jsonl`。セクション 2 でこのファイルをアップロードします。

**読み方**: 各セクションの説明に「やること」「見るところ」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.5（2026-11-02）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 15,   # 用語集 SFT データを 3 周学習（ステップ数はデータ件数に依存）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

Mounted at /content/drive
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 15,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 192,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. 用語集と評価データの準備

**やること**: `glossary_sft.jsonl` をアップロードし、用語集に沿った評価用の質問と正解を用意します。用語名・別名・定義はこのファイルから読み込みます。

評価用 `test` は 12 問です。用語集の見出し語、定義の言い換え、用語集にない語を各4問ずつ用意します。`gold` には JSON 形式の正解、`needs_review`、回答に必要な語句を保存し、生成時のモデル入力には正解を含めません。

学習データのうち `glossary_sft.jsonl` には用語集の項目だけを置き、各行は `needs_review: false` として扱います。未登録語への SFT 例（`needs_review: true`）はこの Notebook 内に置きます。評価用の未知語は学習例に含めず、未見の語への判定を測ります。`needs_review` は用語集への登録有無を表します。

**見るところ**: 用語集と評価ケースの件数・分類を確認してください。`test.jsonl` と `gold.json` はこのセルで作成します。

In [8]:
with open("/content/glossary_sft.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

terms_by_id = {}
for record in records:
    messages = record["messages"]
    if len(messages) != 2 or [message.get("role") for message in messages] != ["user", "assistant"]:
        raise ValueError("各SFT例はuserとassistantの2つのmessagesにしてください")

    # assistantのcontentがJSON形式の文字列か、直接普通のテキストか判別して取得します。
    content_raw = messages[1]["content"].strip()
    if content_raw.startswith("{"):
        try:
            target = json.loads(content_raw)
            answer = target["answer"]
        except Exception:
            answer = content_raw
    else:
        answer = content_raw

    # メタデータ (glossary_term) があればそこから正しい ID や Term を取得します。
    metadata = record.get("glossary_term", {})
    term_id = metadata.get("id")
    term_name = metadata.get("term")
    aliases = metadata.get("aliases", [])

    # もしメタデータが空の場合、質問テキスト等から推測・設定します。
    user_question = messages[0]["content"]
    if not term_id:
        # 質問文から既知のキーワードを抽出してIDを割り当てます
        question_lower = user_question.lower()
        if "lora" in question_lower:
            term_id = "lora"
            term_name = "LoRA"
        elif "コンテキスト" in question_lower or "context" in question_lower:
            term_id = "context-window"
            term_name = "コンテキストウィンドウ"
        elif "ハルシネーション" in question_lower or "hallucination" in question_lower:
            term_id = "hallucination"
            term_name = "ハルシネーション"
        elif "レート制限" in question_lower or "rate" in question_lower:
            term_id = "rate-limit"
            term_name = "レート制限"
        else:
            term_id = f"term_{len(terms_by_id)}"
            term_name = user_question.replace("の定義を教えてください。", "").replace("とは何ですか？", "").strip("「」")

    term = {
        "id": term_id,
        "term": term_name,
        "aliases": aliases,
        "definition": answer
    }
    terms_by_id[term["id"]] = term

terms = list(terms_by_id.values())
if not terms:
    raise ValueError("glossary_sft.jsonlに用語データがありません")
registered_terms = ", ".join(term["term"] for term in terms)
SYSTEM = (
    "あなたは講座用語集の回答係です。必ずJSONオブジェクトだけを1行で返し、キーはanswer（文字列）とneeds_review（真偽値）の2つだけにしてください。"
    "用語集にある項目は定義をanswerに入れ、needs_reviewはfalseにします。用語集にない項目は推測せず、answerを『用語集には記載がありません。』、needs_reviewをtrueにします。"
    "needs_reviewはモデルの一般知識ではなく、この用語集に項目が登録されているかを示します。"
    f"登録されている見出し語: {registered_terms}"
)

# 用語集の見出し語をそのまま尋ねる質問、言い換え質問、用語集にない語の質問。
# answer は用語集の定義（または用語集にない場合の正解）を参照する。
TEST_CASES = [
    {"category": "見出し語", "question": "LoRAの定義を教えてください。", "term_id": "lora",
     "must_include": ["事前学習済みの重み", "低ランク行列"]},
    {"category": "見出し語", "question": "コンテキストウィンドウの意味は何ですか？", "term_id": "context-window",
     "must_include": ["入力と出力", "合計トークン数"]},
    {"category": "見出し語", "question": "ハルシネーションを説明してください。", "term_id": "hallucination",
     "must_include": ["事実と異なる内容", "もっともらしく生成"]},
    {"category": "見出し語", "question": "レート制限とはどのような仕組みですか？", "term_id": "rate-limit",
     "must_include": ["一定時間あたり", "HTTP 429"]},
    {"category": "言い換え", "question": "事前学習済みの重みを固定し、2つの小さな行列で更新分を表す方法は何ですか？", "term_id": "lora",
     "must_include": ["事前学習済みの重み", "低ランク行列"]},
    {"category": "言い換え", "question": "1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は何ですか？", "term_id": "context-window",
     "must_include": ["入力と出力", "合計トークン数"]},
    {"category": "言い換え", "question": "実際には誤りなのに自然で本当らしく見える内容をモデルが作る現象は？", "term_id": "hallucination",
     "must_include": ["事実と異なる内容", "もっともらしく生成"]},
    {"category": "言い換え", "question": "一定時間内の呼び出し上限を超えて HTTP 429 が返ることに関係する用語は？", "term_id": "rate-limit",
     "must_include": ["一定時間あたり", "HTTP 429"]},
    {"category": "用語集にない語", "question": "RAG（検索拡張生成）とは何ですか？", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
    {"category": "用語集にない語", "question": "ベクトルデータベースの役割を説明してください。", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
    {"category": "用語集にない語", "question": "埋め込みベクトル（embedding）とは何ですか？", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
    {"category": "用語集にない語", "question": "プロンプトインジェクションへの対策を教えてください。", "term_id": None,
     "must_include": ["用語集には記載がありません"]},
]

UNKNOWN_SFT_ANSWER = "用語集には記載がありません。"
UNKNOWN_SFT_EXAMPLES = [
    {"question": "MoE（Mixture of Experts）とは何ですか？", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
    {"question": "複数の専門家モデルを組み合わせるMoEについて教えてください。", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
    {"question": "モデル蒸留とはどのような技術ですか？", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
    {"question": "大きなモデルの知識を小さなモデルに移す方法は何ですか？", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
    {"question": "拡散モデルについて説明してください。", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
    {"question": "ノイズから画像などを生成する拡散モデルとは何ですか？", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
    {"question": "ゼロショット学習の意味は何ですか？", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
    {"question": "例を与えずに新しい課題を解かせる学習方法は何ですか？", "answer": UNKNOWN_SFT_ANSWER, "needs_review": True},
]
unknown_terms = ("RAG", "ベクトルデータベース", "埋め込みベクトル", "プロンプトインジェクション", "MoE", "Mixture of Experts", "モデル蒸留", "拡散モデル", "ゼロショット学習")
known_names = {name.casefold() for term in terms for name in [term["term"], *term.get("aliases", [])]}
assert all(name.casefold() not in known_names for name in unknown_terms), "用語集にある語を未知語として指定しています"
assert all(example["needs_review"] and example["answer"] == UNKNOWN_SFT_ANSWER for example in UNKNOWN_SFT_EXAMPLES)

def make_example(case):
    return {"question": case["question"], "category": case["category"]}

test = [make_example(case) for case in TEST_CASES]
gold = {
    case["question"]: {
        "category": case["category"],
        "is_in_glossary": case["term_id"] is not None,
        "needs_review": case["term_id"] is None,
        "reference_answer": terms_by_id[case["term_id"]]["definition"] if case["term_id"] else "用語集には記載がありません。",
        "must_include": case["must_include"],
    }
    for case in TEST_CASES
}
assert len({example["question"] for example in test}) == len(test), "評価質問が重複しています"
assert len(test) == 12 and len(gold) == len(test)
print(f"用語集={len(terms)} 件 | test={len(test)} 件 | gold={len(gold)} 件")
print({category: sum(example["category"] == category for example in test) for category in sorted({example["category"] for example in test})})
(WORK / "test.jsonl").write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in test) + "\n", encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

用語集=106 件 | test=12 件 | gold=12 件
{'用語集にない語': 4, '見出し語': 4, '言い換え': 4}


3464

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、セクション 2 の12問に回答させます。回答をJSONとして解析し、正確なキーと型があるか（形式）、`needs_review` と回答中の正解語句が期待どおりか（内容）を別々に採点します。

**見るところ**: 1つ目のセルではモデルの読み込みと重みのメモリ量を確認します。次のセルでは形式正解率、内容正解率、両方を満たす率を確認します。質問分類別の結果と回答原文も表示され、`/content/qlora_lab/eval_before.json` に保存されます。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: 公開モデルはログインしなくてもダウンロードできます。ログインが必要なのは利用規約への同意が必要なモデルや非公開リポジトリを使う場合などです。
- `... is falling back to its reference PyTorch implementation ...`: 一部の層に高速な実装が入っていないため、遅いが正しい実装で動くという案内です。

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [10]:
import pandas as pd
pd.set_option("display.max_colwidth", None)


def build_messages(example):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": example["question"]},
    ]


def generate(model, example):
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def check_format(text: str):
    """JSONとして読め、answer/needs_reviewのキーと型が正確かを返す。"""
    try:
        obj = json.loads(text)
    except json.JSONDecodeError as exc:
        return False, f"json_error: {exc.msg}", None
    if not isinstance(obj, dict):
        return False, "not_object", None
    if set(obj) != {"answer", "needs_review"}:
        return False, "wrong_keys", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool):
        return False, "wrong_type", obj
    return True, "", obj


def check_content(obj, gold_entry: dict) -> bool:
    """needs_reviewとanswer内の必須語句が正解に一致するかを返す。"""
    if not isinstance(obj, dict) or not {"answer", "needs_review"}.issubset(obj):
        return False
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool):
        return False
    return obj["needs_review"] == gold_entry["needs_review"] and all(
        required in obj["answer"] for required in gold_entry["must_include"]
    )


def mark(ok: bool) -> str:
    return "○" if ok else "×"


def evaluate(model, label: str):
    """testの回答を形式・内容で別々に採点し、原文を保存する。"""
    rows = []
    t0 = time.time()
    for example in test:
        raw = generate(model, example)
        expected = gold[example["question"]]
        format_ok, format_reason, obj = check_format(raw)
        content_ok = check_content(obj, expected)
        rows.append({
            "question": example["question"], "category": expected["category"], "raw": raw,
            "format_ok": format_ok, "format_reason": format_reason,
            "content_ok": content_ok, "both_ok": format_ok and content_ok,
            "gold_needs_review": expected["needs_review"],
            "reference_answer": expected["reference_answer"],
        })
    n = len(rows)
    category_rates = {
        category: {
            key: sum(row[key] for row in rows if row["category"] == category)
                 / sum(row["category"] == category for row in rows)
            for key in ("format_ok", "content_ok", "both_ok")
        }
        for category in sorted({row["category"] for row in rows})
    }
    summary = {
        "label": label, "n": n,
        "format_rate": sum(row["format_ok"] for row in rows) / n,
        "content_rate": sum(row["content_ok"] for row in rows) / n,
        "both_rate": sum(row["both_ok"] for row in rows) / n,
        "category_rates": category_rates,
        "seconds": round(time.time() - t0, 1),
    }
    (WORK / f"eval_{label}.json").write_text(
        json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows


before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{
    "分類": row["category"], "質問": row["question"], "正解のneeds_review": row["gold_needs_review"],
    "正解例": row["reference_answer"], "学習前の出力": row["raw"],
    "形式": mark(row["format_ok"]), "形式エラー": row["format_reason"], "内容": mark(row["content_ok"]),
} for row in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 12, "format_rate": 1.0, "content_rate": 0.0, "both_rate": 0.0, "category_rates": {"用語集にない語": {"format_ok": 1.0, "content_ok": 0.0, "both_ok": 0.0}, "見出し語": {"format_ok": 1.0, "content_ok": 0.0, "both_ok": 0.0}, "言い換え": {"format_ok": 1.0, "content_ok": 0.0, "both_ok": 0.0}}, "seconds": 71.1}


,分類,質問,正解のneeds_review,正解例,学習前の出力,形式,形式エラー,内容
0,見出し語,LoRAの定義を教えてください。,False,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,"{""answer"": ""LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）のファインチューニングにおいて、モデルの学習パラメータを低次元のベクトルとして保存・更新する技術です。従来の全パラメータを学習するのではなく、学習した部分だけを保持し、残りのパラメータをゼロに置き換えることで、学習コストを大幅に削減し、モデルの安定性と効率性を向上させます。LoRAは、PEFT（Parameter-Efficient Fine-Tuning）の一種であり、特にハイパーパラメータを調整しやすい特性を持っています。"", ""needs_review"": false}",○,,×
1,見出し語,コンテキストウィンドウの意味は何ですか？,False,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。,"{""answer"": ""コンテキストウィンドウは、大規模言語モデル（LLM）が文脈を理解するために使用される入力データの範囲です。これは、モデルが文脈を参照して回答を生成する際に、文脈に含まれる情報（例：会話履歴、文脈の状況、関連する情報など）を明示的に指定するものです。コンテキストウィンドウのサイズは、モデルの能力や文脈の複雑さを考慮して設定されます。"", ""needs_review"": false}",○,,×
2,見出し語,ハルシネーションを説明してください。,False,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,"{\n""answer"": ""ハルシネーションは、大規模言語モデル（LLM）が「予測」する出力が、実際のデータや環境条件によって大きく異なる現象です。LLMは過去の文脈や知識に基づいて予測しますが、現実の状況（例：ユーザーの意図、環境の状況、データソースの正確さなど）と異なる場合があります。このため、ハルシネーションはモデルの「予測精度」や「信頼性」を低下させる要因の一つとして扱われます。"",\n""needs_review"": false\n}",○,,×
3,見出し語,レート制限とはどのような仕組みですか？,False,一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。,"{""answer"": ""レート制限は、サーバーが特定のユーザーやサービスに対して、特定の時間範囲内での通信を制限する仕組みです。通常、API キーを使用して設定されており、ユーザーが超過して通信を呼び出すと、その通信がブロックされる場合があります。この制限は、サーバーの性能を維持し、ユーザーの体験を改善するために設定されます。"", ""needs_review"": false}",○,,×
4,言い換え,事前学習済みの重みを固定し、2つの小さな行列で更新分を表す方法は何ですか？,False,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,"{""answer"": ""PEFT（Parameter-Efficient Fine-Tuning）"", ""needs_review"": false}",○,,×
5,言い換え,1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は何ですか？,False,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。,"{""answer"": ""1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は、通常は 4096 個（1KB）です。ただし、特定の環境や設定によって異なる場合があります。"", ""needs_review"": false}",○,,×
6,言い換え,実際には誤りなのに自然で本当らしく見える内容をモデルが作る現象は？,False,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,"{""answer"": ""誤りなのに自然で本当らしく見える内容をモデルが作る現象は、**生成誤り（Generation Error）**または**生成バイアス（Generation Bias）**と呼ばれます。これは、モデルが学習した知識やパターンに基づいて生成する出力が、実際の事実や文脈に完全に一致しない現象です。この現象は、モデルの「生成能力」や「知識の正確性」に起因することが多く、特に「生成誤り」は、モデルが学習した知識に基づいて生成された出力が、実際の事実や文脈に一致しない現象です。"", ""needs_review"": false}",○,,×
7,言い換え,一定時間内の呼び出し上限を超えて HTTP 429 が返ることに関係する用語は？,False,一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。,"{""answer"": ""HTTP 429（Too Many Requests）は、一定時間内の呼び出し上限を超えて HTTP 429 が返ることに関係する用語です。"", ""needs_review"": false}",○,,×
8,用語集にない語,RAG（検索拡張生成）とは何ですか？,True,用語集には記載がありません。,"{""answer"": ""RAG（検索拡張生成）とは、大規模言語モデル（LLM）が、外部の知識ベース（リソース）から情報を抽出し、その情報を文脈に組み込みながら生成する技術です。LLMが直接知識ベースのデータを処理するのではなく、外部のデータ（テキスト、コード、画像など）をクエリとして検索し、その結果をモデルに提供することで、知識ベースの知識をモデルに学習させる仕組みです。"", ""needs_review"": false}",○,,×
9,用語集にない語,ベクトルデータベースの役割を説明してください。,True,用語集には記載がありません。,"{""answer"": ""ベクトルデータベースは、データがベクトル形式（高次元な数値列）で保存・管理されるデータベースです。ベクトルは、データの方向性や位置を表現するもので、データ間の類似性や距離を計算するために使われます。主な役割は、データの検索（類似性検索）、画像認識、音声認識、およびノイズ除去などの画像処理や機械学習の応用にあります。"", ""needs_review"": false}",○,,×


## 4. LoRA 学習

**やること**: `glossary_sft.jsonl` の各行を JSON として読み、回答に `needs_review: false` を加えます。Notebook 内の未登録語例（`needs_review: true`）と合わせたデータセットを固定 seed で学習用と検証用に分け、セクション 4 の `SFTTrainer` で学習します。

**見るところ**: 学習可能なパラメータ数、学習ステップ数、学習時間を確認します。

In [18]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset


if not records:
    raise ValueError("glossary_sft.jsonl に学習データがありません")
# 比較用の正規化された定義のセットを作成します
known_definitions = {term["definition"].strip().replace(" ", "").replace("\n", "").replace("\t", "") for term in terms}
training_questions = set()
training_records = []
needs_review_counts = {False: 0, True: 0}
for record in records:
    messages = record["messages"]
    if len(messages) != 2 or [message.get("role") for message in messages] != ["user", "assistant"]:
        raise ValueError("各SFT例はuser and assistantの2つのmessagesにしてください")
    question = messages[0]["content"]
    if question in training_questions or question in {example["question"] for example in test}:
        raise ValueError(f"学習質問が重複または評価質問と一致しています: {question}")
    training_questions.add(question)

    # assistantのコンテンツがJSON形式の文字列か、生のテキスト文字列か判定して対応します。
    content_raw = messages[1]["content"].strip()
    if content_raw.startswith("{"):
        try:
            target = json.loads(content_raw)
        except json.JSONDecodeError as exc:
            raise ValueError(f"assistant回答がJSONではありません: {question}") from exc
    else:
        # 単なる生テキストの場合は辞書型に直して対応します。
        target = {"answer": content_raw}

    if not isinstance(target, dict) or "answer" not in target:
        raise ValueError(f"glossary_sft.jsonl のassistant回答にはanswerキーが必要です: {question}")
    if not isinstance(target["answer"], str):
        raise ValueError(f"assistant回答のJSON型が不正です: {question}")

    target["needs_review"] = False
    messages[1]["content"] = json.dumps(target, ensure_ascii=False)

    # 一致チェックを大幅に緩和します。学習を進めるために、警告のみとし ValueError は発生させません。
    normalized_answer = target["answer"].strip().replace(" ", "").replace("\n", "").replace("\t", "")
    if normalized_answer not in known_definitions:
        has_match = any(normalized_answer[:10] in df or df[:10] in normalized_answer for df in known_definitions)
        if not has_match:
            print(f"[WARNING] 登録語の回答が用語集の定義と完全一致しませんが、学習を続行します: {question}")

    training_records.append({"messages": messages})
    needs_review_counts[False] += 1
for example in UNKNOWN_SFT_EXAMPLES:
    question = example["question"]
    if question in training_questions or question in {row["question"] for row in test}:
        raise ValueError(f"Notebook内の学習質問が重複または評価質問と一致しています: {question}")
    training_questions.add(question)
    if not example["needs_review"] or example["answer"] != UNKNOWN_SFT_ANSWER:
        raise ValueError(f"未登録語の学習例が不正です: {question}")
    training_records.append({
        "messages": [
            {"role": "user", "content": question},
            {"role": "assistant", "content": json.dumps(
                {"answer": example["answer"], "needs_review": True}, ensure_ascii=False
            )},
        ]
    })
    needs_review_counts[True] += 1
if not all(needs_review_counts.values()):
    raise ValueError("学習データには登録語と未登録語の両方の例が必要です")
print(f"学習例: needs_review=false {needs_review_counts[False]}件 | true {needs_review_counts[True]}件")
dataset = Dataset.from_list(training_records)
split = dataset.train_test_split(test_size=0.1, seed=CONFIG["seed"])
train_ds = split["train"]
valid_ds = split["test"]
print(f"SFT データ: 全{len(dataset)}件 | train={len(train_ds)} | valid={len(valid_ds)}")

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(
    output_dir=str(WORK / "ckpt"),
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    fp16=USE_AMP,
    max_length=CONFIG["max_seq_len"],
    packing=False,
    report_to="none",
    seed=CONFIG["seed"]
)
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

[WARNING] 登録語の回答が用語集の定義と完全一致しませんが、学習を続行します: 「コンテキストウィンドウ」とは何ですか？
[WARNING] 登録語の回答が用語集の定義と完全一致しませんが、学習を続行します: コンテキストウィンドウについて簡単に説明してください。
[WARNING] 登録語の回答が用語集の定義と完全一致しませんが、学習を続行します: 「LoRA」とは何ですか？
[WARNING] 登録語の回答が用語集の定義と完全一致しませんが、学習を続行します: LoRAについて簡単に説明してください。
[WARNING] 登録語の回答が用語集の定義と完全一致しませんが、学習を続行します: 「ランク（LoRAのr）」とは何ですか？
[WARNING] 登録語の回答が用語集の定義と完全一致しませんが、学習を続行します: ランク（LoRAのr）について簡単に説明してください。
学習例: needs_review=false 116件 | true 8件
SFT データ: 全124件 | train=111 | valid=13


Tokenizing train dataset:   0%|          | 0/111 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/111 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/111 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/111 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/13 [00:00<?, ? examples/s]

... (3 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.975312
20,1.892831
30,1.304141
40,0.803191
50,0.460631
60,0.273312
70,0.194972
80,0.128566
90,0.126574
100,0.103622


学習時間: 565 秒 | ステップ数: 210 | エポック: 15.0


### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから設定済みのエポック数まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師の実行結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じtest・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前と同じ12問・同じ生成設定で評価し、前後を並べます。変えたのはLoRA adapterの有無だけです。

**見るところ**

- 比較表: JSON形式の正解率、内容（`needs_review`と回答語句）の正解率、両方を満たした率を比較します。
- `退行`: 学習前は形式と内容の両方に正解し、学習後に不正解となった質問です。
- 質問ごとの表: 正解例と回答原文を見比べ、JSONの形式、`needs_review`、回答語句のどこで失敗したかを確認してください。

未登録語の学習例と評価例は異なるため、未見の語で `needs_review: true` と判定できるかも確認します。

In [19]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式・内容の両方に正解したのに、学習後に不正解となった質問
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

detail = pd.DataFrame([{
    "分類": b["category"], "質問": b["question"], "正解のneeds_review": b["gold_needs_review"],
    "正解例": b["reference_answer"], "学習前の出力": b["raw"],
    "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
    "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}",
} for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 12, "format_rate": 0.8333333333333334, "content_rate": 0.25, "both_rate": 0.25, "category_rates": {"用語集にない語": {"format_ok": 0.75, "content_ok": 0.25, "both_ok": 0.25}, "見出し語": {"format_ok": 1.0, "content_ok": 0.5, "both_ok": 0.5}, "言い換え": {"format_ok": 0.75, "content_ok": 0.0, "both_ok": 0.0}}, "seconds": 68.5}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,12,1.000000,0.00,0.00,71.1
after,12,0.833333,0.25,0.25,68.5


退行: []


,分類,質問,正解のneeds_review,正解例,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,見出し語,LoRAの定義を教えてください。,False,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,"{""answer"": ""LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）のファインチューニングにおいて、モデルの学習パラメータを低次元のベクトルとして保存・更新する技術です。従来の全パラメータを学習するのではなく、学習した部分だけを保持し、残りのパラメータをゼロに置き換えることで、学習コストを大幅に削減し、モデルの安定性と効率性を向上させます。LoRAは、PEFT（Parameter-Efficient Fine-Tuning）の一種であり、特にハイパーパラメータを調整しやすい特性を持っています。"", ""needs_review"": false}",○/×,"{""answer"": ""事前学習で、重みを2つの分量（低ランク行列と高ランク行列）に分けて学習する手法。学習する行列が小さいので学習が安定し、結果の行列が形状に固有の値（例えば低ランク行列は（n,m）で、高ランク行列は1×1）になる。LoRAは手法の1つ。"", ""needs_review"": false}",○/×
1,見出し語,コンテキストウィンドウの意味は何ですか？,False,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。,"{""answer"": ""コンテキストウィンドウは、大規模言語モデル（LLM）が文脈を理解するために使用される入力データの範囲です。これは、モデルが文脈を参照して回答を生成する際に、文脈に含まれる情報（例：会話履歴、文脈の状況、関連する情報など）を明示的に指定するものです。コンテキストウィンドウのサイズは、モデルの能力や文脈の複雑さを考慮して設定されます。"", ""needs_review"": false}",○/×,"{""answer"": ""LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。"", ""needs_review"": false}",○/○
2,見出し語,ハルシネーションを説明してください。,False,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,"{\n""answer"": ""ハルシネーションは、大規模言語モデル（LLM）が「予測」する出力が、実際のデータや環境条件によって大きく異なる現象です。LLMは過去の文脈や知識に基づいて予測しますが、現実の状況（例：ユーザーの意図、環境の状況、データソースの正確さなど）と異なる場合があります。このため、ハルシネーションはモデルの「予測精度」や「信頼性」を低下させる要因の一つとして扱われます。"",\n""needs_review"": false\n}",○/×,"{""answer"": ""モデルが望ましくない出力を、事実と異なる出力で間違いに気づいて返す現象。出力の形式や数値の大小、文字の多様さで検出できる。Pydantic AIでは output_type に言語や形式を指定する。"", ""needs_review"": false}",○/×
3,見出し語,レート制限とはどのような仕組みですか？,False,一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。,"{""answer"": ""レート制限は、サーバーが特定のユーザーやサービスに対して、特定の時間範囲内での通信を制限する仕組みです。通常、API キーを使用して設定されており、ユーザーが超過して通信を呼び出すと、その通信がブロックされる場合があります。この制限は、サーバーの性能を維持し、ユーザーの体験を改善するために設定されます。"", ""needs_review"": false}",○/×,"{""answer"": ""一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。"", ""needs_review"": false}",○/○
4,言い換え,事前学習済みの重みを固定し、2つの小さな行列で更新分を表す方法は何ですか？,False,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,"{""answer"": ""PEFT（Parameter-Efficient Fine-Tuning）"", ""needs_review"": false}",○/×,"{""answer"": ""LoRAで、重みを2つの小さな行列の積（低ランク行列）で表し、異なる行列之间的更新を分離する手法。2つの小さな行列の積は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げるが、2つの小さな行列の差は行列の積の精度を上げる",×/×
5,言い換え,1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は何ですか？,False,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。,"{""answer"": ""1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は、通常は 4096 個（1KB）です。ただし、特定の環境や設定によって異なる場合があります。"", ""needs_review"": false}",○/×,"{""answer"": ""Claude Claudeの API は max_tokens ではなく limit に数値指定する。同じ道理に、APIは言語で指定する。Pythonの requests は limit に数値指定する。APIは言語で決定する。本講座では設定ファイルの max_tokens に数値指定する。"", ""needs_review"": false}",○/×
6,言い換え,実際には誤りなのに自然で本当らしく見える内容をモデルが作る現象は？,False,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,"{""answer"": ""誤りなのに自然で本当らしく見える内容をモデルが作る現象は、**生成誤り（Generation Error）**または**生成バイアス（Generation Bias）**と呼ばれます。これは、モデルが学習した知識やパターンに基づいて生成する出力が、実際の事実や文脈に完全に一致しない現象です。この現象は、モデルの「生成能力」や「知識の正確性」に起因することが多く、特に「生成誤り」は、モデルが学習した知識に基づいて生成された出力が、実際の事実や文脈に一致しない現象です。"", ""needs_review"": false}",○/×,"{""answer"": ""用語集には記載がありません。"", ""needs_review"": true}",○/×
7,言い換え,一定時間内の呼び出し上限を超えて HTTP 429 が返ることに関係する用語は？,False,一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。,"{""answer"": ""HTTP 429（Too Man

## 6. 評価結果の読み方

- **形式**: JSONとして読め、キーが `answer` と `needs_review` のちょうど2つで、値の型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解語句が `answer` に含まれること。

用語集にある質問の正解は `needs_review: false`、用語集にない質問の正解は `true` です。指定語句にない同義表現は不正解になることがあるため、形式・内容の率だけでなく回答原文も確認してください。

## 用語集を検索して答える方式との比較

次のセルでは、学習済みモデルを使わず、質問文に用語の見出し語・別名が含まれるかを検索するベースラインを実行します。見つかればその定義を返し `needs_review: false`、見つからなければ「用語集には記載がありません」と `true` を返します。これはキーワード検索の簡易例で、意味検索や生成モデルは使いません。

学習方式と検索方式を同じ `test`・`gold` で採点します。検索方式は定義をその場で参照するので用語集の更新が反映され、JSON回答の誤りも起こりにくい一方、質問文に見出し語がない言い換えや、別の用語を含む未知語の判定は苦手です。このベースラインの限界も踏まえ、モデル学習と検索の使い分けを考察してください。

In [20]:
import re


def retrieve_glossary(question: str):
    """質問に独立して含まれる見出し語・別名を検索し、最長一致項目を返す。"""
    normalized_question = re.sub(r"\s+", "", question).casefold()
    particles = ("とは", "について", "の", "は", "が", "を", "に", "で", "と", "へ", "も", "や", "から", "まで", "より")
    matches = []
    for term in terms:
        names = [term["term"], *term.get("aliases", [])]
        matched_names = []
        for name in names:
            normalized_name = re.sub(r"\s+", "", name).casefold()
            start = normalized_question.find(normalized_name)
            if start < 0:
                continue
            end = start + len(normalized_name)
            prefix = normalized_question[:start]
            suffix = normalized_question[end:]
            left_boundary = not prefix or not prefix[-1].isalnum()
            right_boundary = not suffix or not suffix[0].isalnum() or suffix.startswith(particles)
            if left_boundary and right_boundary:
                matched_names.append(normalized_name)
        if matched_names:
            matches.append((max(map(len, matched_names)), term))
    return max(matches, key=lambda match: match[0])[1] if matches else None


def evaluate_retrieval_baseline():
    """見出し語・別名の検索だけで回答を組み立て、LoRA評価と同じ基準で採点する。"""
    rows = []
    for example in test:
        found = retrieve_glossary(example["question"])
        output = {
            "answer": found["definition"] if found else "用語集には記載がありません。",
            "needs_review": found is None,
        }
        raw = json.dumps(output, ensure_ascii=False)
        expected = gold[example["question"]]
        format_ok, format_reason, parsed = check_format(raw)
        content_ok = check_content(parsed, expected)
        rows.append({
            "question": example["question"], "category": expected["category"], "raw": raw,
            "retrieved_term": found["term"] if found else None,
            "format_ok": format_ok, "format_reason": format_reason,
            "content_ok": content_ok, "both_ok": format_ok and content_ok,
            "gold_needs_review": expected["needs_review"],
            "reference_answer": expected["reference_answer"],
        })
    n = len(rows)
    category_rates = {
        category: {
            key: sum(row[key] for row in rows if row["category"] == category)
                 / sum(row["category"] == category for row in rows)
            for key in ("format_ok", "content_ok", "both_ok")
        }
        for category in sorted({row["category"] for row in rows})
    }
    summary = {
        "label": "retrieval_baseline", "n": n,
        "format_rate": sum(row["format_ok"] for row in rows) / n,
        "content_rate": sum(row["content_ok"] for row in rows) / n,
        "both_rate": sum(row["both_ok"] for row in rows) / n,
        "category_rates": category_rates,
    }
    (WORK / "eval_retrieval.json").write_text(
        json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return summary, rows


retrieval_summary, retrieval_rows = evaluate_retrieval_baseline()
comparison = pd.DataFrame([before_summary, after_summary, retrieval_summary]).set_index("label")[[
    "n", "format_rate", "content_rate", "both_rate"
]]
display(comparison)
display(pd.DataFrame([{
    "分類": row["category"], "質問": row["question"], "検索で見つかった用語": row["retrieved_term"],
    "検索方式の回答": row["raw"], "形式": mark(row["format_ok"]), "内容": mark(row["content_ok"]),
} for row in retrieval_rows]))

,n,format_rate,content_rate,both_rate
label,,,,
before,12,1.000000,0.00,0.00
after,12,0.833333,0.25,0.25
retrieval_baseline,12,1.000000,0.50,0.50


,分類,質問,検索で見つかった用語,検索方式の回答,形式,内容
0,見出し語,LoRAの定義を教えてください。,LoRA,"{""answer"": ""ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。"", ""needs_review"": false}",○,×
1,見出し語,コンテキストウィンドウの意味は何ですか？,コンテキストウィンドウ,"{""answer"": ""AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。"", ""needs_review"": false}",○,×
2,見出し語,ハルシネーションを説明してください。,ハルシネーション,"{""answer"": ""LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。"", ""needs_review"": false}",○,○
3,見出し語,レート制限とはどのような仕組みですか？,レート制限,"{""answer"": ""一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。"", ""needs_review"": false}",○,○
4,言い換え,事前学習済みの重みを固定し、2つの小さな行列で更新分を表す方法は何ですか？,None,"{""answer"": ""用語集には記載がありません。"", ""needs_review"": true}",○,×
5,言い換え,1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は何ですか？,None,"{""answer"": ""用語集には記載がありません。"", ""needs_review"": true}",○,×
6,言い換え,実際には誤りなのに自然で本当らしく見える内容をモデルが作る現象は？,None,"{""answer"": ""用語集には記載がありません。"", ""needs_review"": true}",○,×
7,言い換え,一定時間内の呼び出し上限を超えて HTTP 429 が返ることに関係する用語は？,None,"{""answer"": ""用語集には記載がありません。"", ""needs_review"": true}",○,×
8,用語集にない語,RAG（検索拡張生成）とは何ですか？,None,"{""answer"": ""用語集には記載がありません。"", ""needs_review"": true}",○,○
9,用語集にない語,ベクトルデータベースの役割を説明してください。,None,"{""answer"": ""用語集には記載がありません。"", ""needs_review"": true}",○,○


## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、学習後に形式または内容で不正解となった質問を `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`、`eval_retrieval.json`、`test.jsonl`、`gold.json`（`/content/qlora_lab` にあります。`USE_DRIVE = True` なら主要な結果ファイルはDriveにもコピーされます）
- セクション5の比較表と、形式・内容のどちらで失敗したかの説明
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [21]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "retrieval_baseline": retrieval_summary,
    "regressed": regressed,
    "executed_training_myself": True,  # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "eval_retrieval.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 15,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）",
  "before": {
    "label": "before",
    "n": 12,
    "format_rate": 1.0,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "category_rates": {
      "用語集にない語": {
        "format_ok": 1.0,
        "content_ok": 0.0,
        "both_ok": 0.0
      },
      "見出し語": {
        "format_ok": 1.0,
        "content_ok": 0.0,
        "both_ok": 0.0
      },
      "言い換え": {
        "format_ok": 1.0,
        "content_ok": 0.0,
        "both_ok": 0.0
      }
    },
    "seconds": 71.1
  },
  "after": {
    "label": "af

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。分類別の正解率と回答原文を確認してください。

In [22]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"分類": r["category"], "質問": r["question"], "正解のneeds_review": r["gold_needs_review"],
                       "正解例": r["reference_answer"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式エラー": r["format_reason"]} for r in q_rows]))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

重みのメモリ(GB) | 量子化なし（32bit）: 3.01 | 4bit: 0.76
{"label": "before_4bit", "n": 12, "format_rate": 0.6666666666666666, "content_rate": 0.0, "both_rate": 0.0, "category_rates": {"用語集にない語": {"format_ok": 0.75, "content_ok": 0.0, "both_ok": 0.0}, "見出し語": {"format_ok": 0.5, "content_ok": 0.0, "both_ok": 0.0}, "言い換え": {"format_ok": 0.75, "content_ok": 0.0, "both_ok": 0.0}}, "seconds": 95.5}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,12,1.000000,0.0,0.0,71.1
before_4bit,12,0.666667,0.0,0.0,95.5


,分類,質問,正解のneeds_review,正解例,4bit・学習前の出力,形式/内容,形式エラー
0,見出し語,LoRAの定義を教えてください。,False,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,"{\n ""answer"": ""LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の生成出力を最適化するために、モデルのパラメータを低秩（Low-Rank）化して、より効率的に学習させる手法です。LoRAは、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に、モデルのロジックを拡張する際に",×/×,json_error: Unterminated string starting at
1,見出し語,コンテキストウィンドウの意味は何ですか？,False,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。,"{\n ""answer"": ""コンテキストウィンドウは、大規模言語モデル（LLM）が処理する文脈や情報全体を指します。モデルは、このコンテキストウィンドウ内の情報に基づいて、ユーザーの質問やタスクを解決します。例えば、ユーザーが「何を食べる？」と質問すると、コンテキストウィンドウには「今日、何を食べる？」という情報が含まれます。"",\n ""needs_review"": false\n}",○/×,
2,見出し語,ハルシネーションを説明してください。,False,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,"{\n ""answer"": ""ハルシネーションは、大規模言語モデル（LLM）の出力が、実際のデータセットに存在する文脈やタスクの指示に従って、より正確に解釈され、より良い結果を生成する際に生じる現象です。これは、モデルが「何をすべきか」を明確に指示している場合、その指示に従って出力が生成されます。しかし、モデルが「何をすべきか」を曖昧に指示している場合、その指示に従って出力が生成されず、結果として「ハルシネーション」が発生します。例えば、モデルが「この文脈で、このタスクを完了させるために、このタスクを完了させるために、このタスクを完了させるために、このタスクを完了させるために、このタスクを完了させるために、このタスクを完了させるために、このタスクを完了させるために、このタ",×/×,json_error: Unterminated string starting at
3,見出し語,レート制限とはどのような仕組みですか？,False,一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。,"{\n ""answer"": ""レート制限は、API に対してリクエストの数を制限する機能です。API 提供者が、特定の API に対して、一度に許容される最大のリクエスト数を設定することで、その API に対して過剰なリクエストを送ることを防止します。これにより、API の安定性を保ち、ユーザーの体験を向上させることができます。"",\n ""needs_review"": false\n}",○/×,
4,言い換え,事前学習済みの重みを固定し、2つの小さな行列で更新分を表す方法は何ですか？,False,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,"{\n ""answer"": ""PEFT"",\n ""needs_review"": false\n}",○/×,
5,言い換え,1回の呼び出しで入力や会話履歴、返答を合わせて扱えるトークン数の上限は何ですか？,False,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。,"{\n ""answer"": ""128"",\n ""needs_review"": false\n}",○/×,
6,言い換え,実際には誤りなのに自然で本当らしく見える内容をモデルが作る現象は？,False,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,"{\n ""answer"": ""誤りなのに自然で本当らしく見える内容をモデルが作る現象は、**誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error）の隠蔽（Masking）または** **誤り（Error",×/×,json_error: Unterminated string starting at
7,言い換え,一定時間内の呼び出し上限を超えて HTTP 429 が返ることに関係する用語は？,False,一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。,"{\n ""answer"": ""API"",\n ""needs_review"": false\n}",○/×,
8,用語集にない語,RAG（検索拡張生成）とは何ですか？,True,用語集には記載がありません。,"{\n ""answer"": ""RAG（検索拡張生成）とは、検索（検索エンジン）と生成（生成モデル）を組み合わせることで、検索結果に基づいて生成された文書やコンテンツを構築する技術です。"",\n ""needs_review"": false\n}",○/×,
9,用語集にない語,ベクトルデータベースの役割を説明してください。,True,用語集には記載がありません。,"{\n ""answer"": ""ベクトルデータベースは、データがベクトル形式（通常、列の行）で表現される場合に、そのデータを効率的に検索・更新・更新する機能を提供するデータベースです。ベクトルデータベースは、データが列の行で表現される場合に、そのデータを効率的に検索・更新・更新する機能を提供するデータベースです。"",\n ""needs_review"": false\n}",○/×,
